In [7]:
!pip install transformers datasets torch accelerate

Defaulting to user installation because normal site-packages is not writeable


In [3]:
# ================================
# GPT-2 Fine-Tuning (Full Script)
# ================================

# Install dependencies (run once in terminal or notebook)
# pip install transformers datasets torch

# -------------------------------
# 1. Import Libraries
# -------------------------------
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset
import torch

# -------------------------------
# 2. Load Pretrained GPT-2 Model & Tokenizer
# -------------------------------
model_name = "gpt2"

# Load tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# GPT-2 does not have a padding token → set it manually
tokenizer.pad_token = tokenizer.eos_token
model.resize_token_embeddings(len(tokenizer))

# -------------------------------
# 3. Load Dataset
# -------------------------------
# Make sure you have a file named "data.txt" in same folder

dataset = load_dataset("text", data_files={"train": "data.txt"})

# -------------------------------
# 4. Tokenization Function
# -------------------------------
def tokenize_function(examples):
    # Convert text into tokens (numbers)
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    # IMPORTANT:
    # For GPT-2, labels must be same as input_ids
    # This enables next-word prediction training
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

# Apply tokenization
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# -------------------------------
# 5. Data Collator (Handles batching correctly)
# -------------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # VERY IMPORTANT → GPT-2 is not masked LM
)

# -------------------------------
# 6. Training Configuration
# -------------------------------
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",   # where model will be saved
    num_train_epochs=3,              # number of training passes
    per_device_train_batch_size=2,   # batch size (keep small for CPU)
    save_steps=500,                 # save checkpoint every 500 steps
    save_total_limit=2,             # keep only last 2 checkpoints
    logging_steps=100,              # print logs every 100 steps
    learning_rate=5e-5,             # learning rate
    report_to="none"                # disables wandb/logging issues
)

# -------------------------------
# 7. Trainer Setup
# -------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator,
)

# -------------------------------
# 8. Train the Model
# -------------------------------
trainer.train()

# -------------------------------
# 9. Save Fine-Tuned Model
# -------------------------------
trainer.save_model("./gpt2-finetuned")
tokenizer.save_pretrained("./gpt2-finetuned")

print("✅ Training complete and model saved!")

# -------------------------------
# 10. Text Generation
# -------------------------------
from transformers import pipeline

# Load fine-tuned model
generator = pipeline(
    "text-generation",
    model="./gpt2-finetuned",
    tokenizer=tokenizer
)

# Input prompt
prompt = "Artificial intelligence in education"

# Generate text
output = generator(
    prompt,
    max_new_tokens=100,
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    num_return_sequences=1
)

# Print result
print("\nGenerated Text:\n")
print(output[0]["generated_text"])


/opt/intel/oneapi/intelpython/lib/python3.11/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss




riting model shards: 100%|█████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.80s/it]

✅ Training complete and model saved!



[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Text:

Artificial intelligence in education is gaining traction in key areas such as healthcare, finance, and education. The development of artificial intelligence has greatly improved the efficiency of learning algorithms, including algorithms that predict patterns that are not necessarily optimal for all learners. In addition, artificial intelligence is increasingly being used for predictive and reinforcement learning, such as reinforcement learning, in education. The use of machine learning to predict learning performance is gaining momentum in this field, with new techniques such as deep learning and reinforcement learning becoming increasingly useful for predictive and reinforcement learning
